In [4]:
import math
import pyBigWig

In [5]:
# Path to a bigWig file to extract chromosome sizes from
reference_bw = "data/GSM6757743_orer_bcd_mbt_nexus_1_negative.bw"  # Replace with one of your real bigWig files

# Load chromosome sizes from the bigWig
with pyBigWig.open(reference_bw) as bw:
    chrom_sizes = bw.chroms()

In [6]:
def round_up_to_multiple(x, base):
    return int(base * math.ceil(x/base))

input_bed = "all_peaks.bed"
output_bed = "all_peaks_200bp.bed"

In [7]:
with open(input_bed) as infile, open(output_bed, "w") as out:
    for line in infile:
        chrom, start, end = line.strip().split("\t")[:3]
        start = int(start)
        end = int(end)

        if chrom not in chrom_sizes:
            continue  # Skip unknown chromosomes

        chrom_len = chrom_sizes[chrom]

        # Original peak length
        original_len = end - start

        # Round to nearest multiple of 200
        rounded_len = round_up_to_multiple(original_len, 200)
        overhang = rounded_len - original_len

        # Tentative expansion
        expand_left = overhang // 2
        expand_right = overhang - expand_left

        new_start = start - expand_left
        new_end = end + expand_right

        # Clamp within chromosome bounds
        if new_start < 0:
            shift = abs(new_start)
            new_start = 0
            new_end = min(chrom_len, new_end + shift)  # shift right to preserve length
        elif new_end > chrom_len:
            shift = new_end - chrom_len
            new_end = chrom_len
            new_start = max(0, new_start - shift)  # shift left to preserve length

        # Split into 200bp windows
        for window_start in range(new_start, new_end, 200):
            window_end = window_start + 200
            if window_end <= new_end:
                out.write(f"{chrom}\t{window_start}\t{window_end}\n")


In [2]:
#To exclude blacklist-overlapping windows, we can use bedtools.
import subprocess

# Paths
blacklist_bed = "gReLU/src/grelu/resources/blacklists/encode/dm6-blacklist.v2.bed"
input_bed = "all_peaks_200bp.bed"
filtered_bed = "filtered_200bp_windows.bed"

# Run bedtools to exclude blacklist-overlapping windows
subprocess.run([
    "bedtools", "intersect",
    "-a", input_bed,
    "-b", blacklist_bed,
    "-v"  # Exclude overlapping windows
], stdout=open(filtered_bed, "w"))

print(f"Filtered BED written to: {filtered_bed}")


Filtered BED written to: filtered_200bp_windows.bed
